In [31]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal, Any, Annotated, List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

load_dotenv()

generator_llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')
evaluator_llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')
optimizer_llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')


In [32]:
class TweetState(TypedDict):
    topic: str
    tweet: str 
    evaluation: Literal['approved', 'needs_improvement']
    feedback: str
    iteration: int 
    max_iteration: int 
    tweet_history: Annotated[List[str], operator.add]
    feedback_history: Annotated[List[str], operator.add]
    

In [33]:
class TweetEvaluationSchema(BaseModel):
    evaluation: Literal['approved', 'needs_improvement'] = Field(..., description='Final evaluation of tweet')
    feedback: str = Field(..., description='Constructive feedback for the tweet.')

In [34]:
def generate_tweet(state: TweetState) -> Any: 
    prompt = f"""
    You are a funny and clever Twitter influencer.
    Write a short, origininal and hilarious tweet on the topic: {state['topic']}.

    Note:
    - Max 250 characters.
    - Use observational humor, irony, sarcasm or cultural references.
    - This is version {state['iteration'] + 1}
    """

    response = generator_llm.invoke(prompt).content
    return {'tweet': response, 'tweet_history': [response]}


def evaluate_tweet(state: TweetState) -> Any:
    prompt = f"""
    You are a strict, twitter critic. You evaluate tweets based on humor, originality, virality and tweet format.

    Evaluate the following tweet: {state['tweet']}
    Respond ONLY in structured format.
    - evaluation: 'approved' or 'needs_improvement'
    - feedback: One paragraph feedback explaining the strengths and weaknesses of tweet.
    """

    structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluationSchema)
    response = structured_evaluator_llm.invoke(prompt)
    return {'evaluation': response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

def optimize_tweet(state: TweetState) -> Any:
    prompt = f"""
    You punch up tweets for virality and humor based on given feedback.
    Improve the tweet based on below: 
    - Original tweet: {state['tweet']}
    - Feedback: {state['feedback']}
    - Topic: {state['topic']}

    Re-write it as a short, viral-worthy tweet. Keep it under 200 characters.

    """
    response = optimizer_llm.invoke(prompt).content
    iteration = state['iteration'] + 1
    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}

In [35]:
def route_evaluation(state: TweetState) -> Any:
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [36]:
graph = StateGraph(TweetState)
graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'optimize'})

graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()

initial_state = {
    'topic': 'Aeroplane pilot',
    'iteration': 1,
    'max_iteration': 5
}
workflow.invoke(initial_state)



{'topic': 'Aeroplane pilot',
 'tweet': 'My pilot just announced, "We\'re cleared for takeoff, and by \'cleared,\' I mean the coffee machine is still smoking but we\'re going for it." Sounds about right. #AeroplaneLife #TrustTheProcess ✈️😂',
 'evaluation': 'approved',
 'feedback': "This tweet successfully uses humor derived from a relatable, albeit slightly concerning, airplane scenario. The pilot's humorous announcement and the user's pithy comment, 'Sounds about right,' create a strong comedic punch. The hashtags #AeroplaneLife and #TrustTheProcess, along with the airplane and laughing emojis, enhance its potential for virality and clearly convey the tweet's theme and tone. The format is concise and engaging, well-suited for Twitter.",
 'iteration': 1,
 'max_iteration': 5,
 'tweet_history': ['My pilot just announced, "We\'re cleared for takeoff, and by \'cleared,\' I mean the coffee machine is still smoking but we\'re going for it." Sounds about right. #AeroplaneLife #TrustTheProcess 